# 00. Project Overview

This notebook defines the scientific scope of the ChemDFM perturbation-response project. The study models drug-induced expression responses in single-cell data by predicting perturbation effects relative to a cell-type-specific control anchor. The primary objective is not merely to optimize latent reconstruction error; rather, it is to determine whether the model preserves biologically meaningful response structure under in-distribution and out-of-distribution evaluation settings.

The repository is organized as a notebook-first research workflow. Each downstream notebook has one responsibility: data audit, split integrity, preprocessing, baseline construction, model training, post-hoc biological evaluation, robustness, statistics, and paper artifact generation. This separation prevents hidden state, improves traceability, and makes each result reproducible from disk.


## Core hypotheses

1. A control-anchored residual model should outperform trivial baselines that ignore perturbation structure.
2. Cell-aware regularization should improve out-of-distribution response fidelity.
3. Latent-space accuracy alone is insufficient; therefore, gene-space differential expression recovery, pathway-profile recovery, and dose-trend diagnostics are required.
4. Claims should be supported not only by a single run, but also by ablations, seed variability, and confidence intervals.


In [4]:
!rm -rf /content/drive

In [5]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

Mounted at /content/drive
Download complete.
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM_repo
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cuda


In [6]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad


In [7]:
project_map = {
    "00_project_overview": "study scope, hypotheses, artifact map",
    "01_data_intake_and_audit": "input audit and integrity checks",
    "02_split_construction_and_leakage_checks": "split validity and leakage analysis",
    "03_preprocessing_and_pca": "train-only PCA, encoders, control anchors",
    "04_baselines": "trivial and structured baseline models",
    "05_model_residual_train": "residual model training",
    "06_model_residual_cellaware_mmd_train": "residual + cell-aware MMD training",
    "07_posthoc_gene_space_evaluation": "gene-space reconstruction and biological diagnostics",
    "08_biological_validation": "external gene-set enrichment validation",
    "09_ablation_and_robustness": "component-wise ablations and sensitivity",
    "10_cross_split_generalization": "multiple split schemes",
    "11_statistics_and_confidence_intervals": "bootstrap CIs and paired comparisons",
    "12_figures_and_tables_for_paper": "paper-ready figures and tables",
    "99_sandbox_experiments": "temporary exploratory work only",
}
pd.DataFrame(
    [{"notebook": k + ".ipynb", "purpose": v} for k, v in project_map.items()]
)


,notebook,purpose
0,00_project_overview.ipynb,"study scope, hypotheses, artifact map"
1,01_data_intake_and_audit.ipynb,input audit and integrity checks
2,02_split_construction_and_leakage_checks.ipynb,split validity and leakage analysis
3,03_preprocessing_and_pca.ipynb,"train-only PCA, encoders, control anchors"
4,04_baselines.ipynb,trivial and structured baseline models
5,05_model_residual_train.ipynb,residual model training
6,06_model_residual_cellaware_mmd_train.ipynb,residual + cell-aware MMD training
7,07_posthoc_gene_space_evaluation.ipynb,gene-space reconstruction and biological diagn...
8,08_biological_validation.ipynb,external gene-set enrichment validation
9,09_ablation_and_robustness.ipynb,component-wise ablations and sensitivity


## Expected canonical artifacts

The canonical raw dataset should be placed at:

`/content/drive/MyDrive/ChemDFM_repo/data/raw/sciplex_complete_middle_subset.h5ad`

Subsequent notebooks create standardized artifacts under `data/interim`, `data/processed`, `runs`, `results`, and `reports`. In particular, train-only PCA objects, encoders, control anchors, baseline metrics, model checkpoints, and gene-space evaluation outputs should be saved to disk and reloaded downstream. The next notebook begins with dataset intake and audit.
